# Lab 2 - Dimensionality Reduction

Requirements:
1. **PCA, Kernel PCA:** Implement PCA manually (using Numpy/SVD) and using Scikit-Learn. Compare the results. Use `sklearn.decomposition.KernelPCA` on a non-linear dataset (e.g., `make_moons`).
2. **Other Dimensionality Reduction Techniques:** Visualize the MNIST or Fashion-MNIST dataset using t-SNE and PCA. Compare the cluster separation.

## 1. PCA and Kernel PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.datasets import make_moons, fetch_openml
from sklearn.manifold import TSNE
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)

### 1.1 Manual PCA using Numpy/SVD

Steps for manual PCA:
1. Center the data (subtract mean).
2. Compute Singular Value Decomposition (SVD).
3. Project the data onto the first $d$ principal components.

In [ ]:
# Generate a random 3D dataset
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1

angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)

# 1. Center the data
X_centered = X - X.mean(axis=0)

# 2. Compute SVD
U, s, Vt = np.linalg.svd(X_centered)

# Get the first two principal components
c1 = Vt.T[:, 0]
c2 = Vt.T[:, 1]

# 3. Project down to 2D
W2 = Vt.T[:, :2]
X2D_manual = X_centered.dot(W2)

print("Shape of manually projected data:", X2D_manual.shape)
print("First 5 rows of manually projected data:\n", X2D_manual[:5])

### 1.2 PCA using Scikit-Learn

Now, we use `sklearn.decomposition.PCA` to perform the same dimensionality reduction.

In [ ]:
pca = PCA(n_components=2)
X2D_sklearn = pca.fit_transform(X)

print("Shape of sklearn projected data:", X2D_sklearn.shape)
print("First 5 rows of sklearn projected data:\n", X2D_sklearn[:5])

### 1.3 Comparison between Manual PCA and Scikit-Learn PCA

Notice that the Scikit-Learn PCA results might have opposite signs for some axes compared to our manual SVD implementation. This is normal because a principal component axis can be pointing in either direction. The variance explained remains the same.

In [ ]:
# Compare the two projections
# The results should be almost equal (or equal but with opposite signs)
print("Are the results almost equal? (Ignoring signs)")
print(np.allclose(np.abs(X2D_manual), np.abs(X2D_sklearn)))

### 1.4 Kernel PCA

Kernel PCA is used to perform complex nonlinear projections. We will test it on the `make_moons` dataset.

In [ ]:
X_moons, y_moons = make_moons(n_samples=100, noise=0.15, random_state=42)

lin_pca = KernelPCA(n_components=2, kernel="linear", fit_inverse_transform=True)
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04, fit_inverse_transform=True)
sig_pca = KernelPCA(n_components=2, kernel="sigmoid", gamma=0.001, coef0=1, fit_inverse_transform=True)

plt.figure(figsize=(15, 4))
for subplot, pca, title in ((131, lin_pca, "Linear kernel"), 
                            (132, rbf_pca, "RBF kernel, $\gamma=0.04$"), 
                            (133, sig_pca, "Sigmoid kernel, $\gamma=10^{-3}, r=1$")):
    X_reduced = pca.fit_transform(X_moons)
    plt.subplot(subplot)
    plt.title(title, fontsize=14)
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y_moons, cmap=plt.cm.coolwarm)
    plt.xlabel("$z_1$", fontsize=18)
    if subplot == 131:
        plt.ylabel("$z_2$", fontsize=18, rotation=0)
    plt.grid(True)

plt.show()

## 2. Other Dimensionality Reduction Techniques: t-SNE vs PCA

We will load a subset of the MNIST dataset (e.g., 2000 samples) and reduce its dimensionality to 2D using both PCA and t-SNE. We will then plot the results to see which algorithm better separates the different digit clusters.

In [ ]:
# Load a subset of MNIST data
mnist = fetch_openml('mnist_784', version='active', parser='auto')
X_mnist = mnist.data
y_mnist = mnist.target.astype(int)

# Select a subset to speed up computation
np.random.seed(42)
subset_indices = np.random.choice(len(X_mnist), 2000, replace=False)
X_subset = X_mnist.iloc[subset_indices] if hasattr(X_mnist, 'iloc') else X_mnist[subset_indices]
y_subset = y_mnist.iloc[subset_indices] if hasattr(y_mnist, 'iloc') else y_mnist[subset_indices]

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
X_pca_reduced = pca.fit_transform(X_subset)

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne_reduced = tsne.fit_transform(X_subset)

In [ ]:
# Plot the results side-by-side
plt.figure(figsize=(16, 6))

plt.subplot(121)
plt.scatter(X_pca_reduced[:, 0], X_pca_reduced[:, 1], c=y_subset, cmap="jet", alpha=0.5)
plt.colorbar()
plt.title("PCA - 2D Projection of MNIST (2000 samples)", fontsize=14)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)

plt.subplot(122)
plt.scatter(X_tsne_reduced[:, 0], X_tsne_reduced[:, 1], c=y_subset, cmap="jet", alpha=0.5)
plt.colorbar()
plt.title("t-SNE - 2D Projection of MNIST (2000 samples)", fontsize=14)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.grid(True)

plt.show()

### Conclusion on t-SNE vs PCA

As we can observe from the plots above:
- **PCA** is a linear dimensionality reduction technique. It tries to preserve the global structure (variance) of the data. However, the different digit clusters in the MNIST dataset are heavily overlapping in the 2D PCA projection.
- **t-SNE** is a non-linear technique designed specifically for visualization. It excels at preserving the local structure, keeping similar instances close to each other. Consequently, it creates much more distinct and well-separated clusters for each digit class.